# Modeling — Revised Time-Series Validation

## Notebook目的

在保留Ridge、SVM、Decision Tree和LightGBM研究逻辑的前提下，建立严格的时间序列Train/Validation/Test流程。Test默认锁定，只允许在模型、特征和阈值全部冻结后进行一次最终评价。

## 输入数据

- `Train_data_revised.pkl`
- `Validation_data_revised.pkl`
- `Test_data_revised.pkl`
- `feature_manifest_revised.csv`

## 输出结果

运行后可生成：

- `validation_metrics_revised.csv`
- `validation_predictions_revised.pkl`
- `lightgbm_feature_importance_revised.csv`
- 在显式开启最终测试后生成 `test_metrics_revised.csv` 与 `test_predictions_revised.pkl`

当前Notebook不包含虚构结果。

## 对应研究文档

- [[../../04_Model/模型体系|模型体系]]
- [[../../04_Model/LightGBM模型说明|LightGBM模型说明]]
- [[../../04_Model/模型评价|模型评价]]
- [[../../01_Research/技术路线|技术路线]]


## 模型研究层级

```text
Linear Regression（建议的基础Baseline，现有项目未形成独立实验）
        ↓
Ridge / Lasso（正则化模型；当前保留Ridge）
        ↓
Decision Tree / Random Forest / SVM（非线性模型；
当前保留Decision Tree与原有RBF-SVR，Random Forest不强行新增实验）
        ↓
LightGBM（Boosting主模型候选）
```

LightGBM是主模型候选，不预设其一定优于其他模型。所有比较先在Validation完成，Test不参与调参、特征筛选或Early Stopping。


In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "PROJECT_STATUS.md").exists():
            return candidate
    raise FileNotFoundError("无法定位项目根目录。")

PROJECT_ROOT = (
    Path(os.environ["ML_FUTURES_PROJECT_ROOT"]).resolve()
    if "ML_FUTURES_PROJECT_ROOT" in os.environ
    else find_project_root()
)
DATA_DIR = PROJECT_ROOT / "02_Data/processed/revised"
OUTPUT_DIR = PROJECT_ROOT / "05_Code/revised/artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "target_return_3s"
RUN_SVM = False  # RBF-SVR在高频大样本上成本较高，需显式开启。
RUN_FINAL_TEST = False  # Test默认锁定。


In [ ]:
required_paths = {
    "train": DATA_DIR / "Train_data_revised.pkl",
    "validation": DATA_DIR / "Validation_data_revised.pkl",
    "test": DATA_DIR / "Test_data_revised.pkl",
    "manifest": DATA_DIR / "feature_manifest_revised.csv",
}
missing_paths = [
    str(path) for path in required_paths.values() if not path.exists()
]
if missing_paths:
    raise FileNotFoundError(
        "请先运行 revised/ML1_homework.ipynb。缺少: "
        + ", ".join(missing_paths)
    )

train_data = pd.read_pickle(required_paths["train"]).sort_values(
    "trade_time"
)
validation_data = pd.read_pickle(
    required_paths["validation"]
).sort_values("trade_time")
test_data = pd.read_pickle(required_paths["test"]).sort_values(
    "trade_time"
)
feature_manifest = pd.read_csv(required_paths["manifest"])
FEATURE_COLUMNS = feature_manifest["feature"].tolist()

assert train_data["trade_time"].max() < validation_data["trade_time"].min()
assert validation_data["trade_time"].max() < test_data["trade_time"].min()
assert not (set(train_data.index) & set(validation_data.index))
assert not (set(validation_data.index) & set(test_data.index))

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]
X_validation = validation_data[FEATURE_COLUMNS]
y_validation = validation_data[TARGET_COLUMN]

for name, matrix in {
    "X_train": X_train,
    "X_validation": X_validation,
}.items():
    values = matrix.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"{name}包含NaN或无穷值。")

len(FEATURE_COLUMNS), X_train.shape, X_validation.shape


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "mse": mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
        "prediction_std": float(np.std(y_pred)),
    }

validation_predictions = validation_data[
    ["trade_time", "trade_date", "session_id", TARGET_COLUMN]
].copy()
validation_metrics = {}
fitted_models = {}


## 保留模型

- **Ridge**：带L2正则的线性基准，使用只在Train上拟合的StandardScaler。
- **RBF-SVR**：原项目已有的非线性模型，默认关闭以避免在大样本上无意触发高成本训练。
- **Decision Tree**：保留原来的深度与特征数逻辑，并增加随机种子。
- **LightGBM**：使用按日期在后的Validation做Early Stopping。

Linear Regression和Random Forest只保留研究定位说明，不在没有原实验依据的情况下强行新增结果。


In [ ]:
ridge_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]
)
ridge_model.fit(X_train, y_train)
ridge_validation_prediction = ridge_model.predict(X_validation)
fitted_models["ridge"] = ridge_model
validation_predictions["pred_ridge"] = ridge_validation_prediction
validation_metrics["ridge"] = regression_metrics(
    y_validation, ridge_validation_prediction
)

tree_model = DecisionTreeRegressor(
    max_depth=5,
    max_features=min(20, len(FEATURE_COLUMNS)),
    random_state=SEED,
)
tree_model.fit(X_train, y_train)
tree_validation_prediction = tree_model.predict(X_validation)
fitted_models["decision_tree"] = tree_model
validation_predictions[
    "pred_decision_tree"
] = tree_validation_prediction
validation_metrics["decision_tree"] = regression_metrics(
    y_validation, tree_validation_prediction
)

if RUN_SVM:
    svm_model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf", C=10)),
        ]
    )
    svm_model.fit(X_train, y_train)
    svm_validation_prediction = svm_model.predict(X_validation)
    fitted_models["svm"] = svm_model
    validation_predictions["pred_svm"] = svm_validation_prediction
    validation_metrics["svm"] = regression_metrics(
        y_validation, svm_validation_prediction
    )


## LightGBM参数原则

下列参数是可复现、偏保守的研究起点，不是为了提高收益而事后挑选：

- 保留原项目的 `learning_rate=0.01`、`num_leaves=31`、特征与样本子采样。
- 增加 `max_depth`、`min_data_in_leaf` 和L2正则以限制Leaf-wise过拟合。
- 增加完整随机种子。
- 最大迭代放宽，实际轮数由按时间在后的Validation Early Stopping决定。
- Test不参与Early Stopping或Feature Importance筛选。


In [ ]:
lgb_params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 31,
    "max_depth": 6,
    "min_data_in_leaf": 200,
    "learning_rate": 0.01,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": SEED,
    "feature_fraction_seed": SEED,
    "bagging_seed": SEED,
    "data_random_seed": SEED,
    "deterministic": True,
    "force_col_wise": True,
}

lgb_train = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=FEATURE_COLUMNS,
    free_raw_data=False,
)
lgb_validation = lgb.Dataset(
    X_validation,
    label=y_validation,
    feature_name=FEATURE_COLUMNS,
    reference=lgb_train,
    free_raw_data=False,
)

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_validation],
    valid_names=["chronological_validation"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ],
)
lgb_validation_prediction = lgb_model.predict(
    X_validation, num_iteration=lgb_model.best_iteration
)
fitted_models["lightgbm"] = lgb_model
validation_predictions["pred_lightgbm"] = lgb_validation_prediction
validation_metrics["lightgbm"] = regression_metrics(
    y_validation, lgb_validation_prediction
)


In [ ]:
validation_metrics_frame = pd.DataFrame(validation_metrics).T
validation_metrics_frame.index.name = "model"
validation_metrics_frame.to_csv(
    OUTPUT_DIR / "validation_metrics_revised.csv"
)
validation_predictions.to_pickle(
    OUTPUT_DIR / "validation_predictions_revised.pkl"
)

feature_importance = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "gain": lgb_model.feature_importance(
            importance_type="gain"
        ),
        "split": lgb_model.feature_importance(
            importance_type="split"
        ),
    }
)
total_gain = feature_importance["gain"].sum()
feature_importance["gain_share"] = np.where(
    total_gain > 0,
    feature_importance["gain"] / total_gain,
    0.0,
)
feature_importance.sort_values(
    ["gain", "split"], ascending=False, inplace=True
)
feature_importance.to_csv(
    OUTPUT_DIR / "lightgbm_feature_importance_revised.csv",
    index=False,
)

top_importance = feature_importance.head(20).sort_values("gain")
top_importance.plot.barh(
    x="feature", y="gain", figsize=(10, 7), legend=False
)
plt.title("LightGBM Feature Importance — Gain (Validation Model)")
plt.xlabel("Total gain")
plt.tight_layout()
plt.show()

validation_metrics_frame


## Feature Importance解释边界

- Gain和Split只描述模型如何使用特征，不代表因果关系。
- 高度相关特征会互相分摊重要性。
- 不根据一次Importance结果立即删除特征。
- 后续应比较不同Walk-forward窗口的重要性稳定性，并在Validation上做分组消融。


In [ ]:
if RUN_FINAL_TEST:
    # 只有在特征、参数和阈值全部冻结后才允许执行本单元。
    train_final = pd.concat(
        [train_data, validation_data], axis=0
    ).sort_values("trade_time")
    X_train_final = train_final[FEATURE_COLUMNS]
    y_train_final = train_final[TARGET_COLUMN]
    X_test = test_data[FEATURE_COLUMNS]
    y_test = test_data[TARGET_COLUMN]
    if not np.isfinite(X_test.to_numpy(dtype=float)).all():
        raise ValueError("X_test包含NaN或无穷值。")

    final_models = {}

    final_ridge = clone(ridge_model)
    final_ridge.fit(X_train_final, y_train_final)
    final_models["ridge"] = final_ridge

    final_tree = clone(tree_model)
    final_tree.fit(X_train_final, y_train_final)
    final_models["decision_tree"] = final_tree

    if RUN_SVM:
        final_svm = clone(svm_model)
        final_svm.fit(X_train_final, y_train_final)
        final_models["svm"] = final_svm

    final_lgb_train = lgb.Dataset(
        X_train_final,
        label=y_train_final,
        feature_name=FEATURE_COLUMNS,
    )
    final_lightgbm = lgb.train(
        lgb_params,
        final_lgb_train,
        num_boost_round=lgb_model.best_iteration,
    )
    final_models["lightgbm"] = final_lightgbm

    test_predictions = test_data[
        ["trade_time", "trade_date", "session_id", TARGET_COLUMN]
    ].copy()
    test_metrics = {}
    for model_name, model in final_models.items():
        if model_name == "lightgbm":
            prediction = model.predict(
                X_test, num_iteration=lgb_model.best_iteration
            )
        else:
            prediction = model.predict(X_test)
        test_predictions[f"pred_{model_name}"] = prediction
        test_metrics[model_name] = regression_metrics(
            y_test, prediction
        )

    test_metrics_frame = pd.DataFrame(test_metrics).T
    test_metrics_frame.index.name = "model"
    test_metrics_frame.to_csv(
        OUTPUT_DIR / "test_metrics_revised.csv"
    )
    test_predictions.to_pickle(
        OUTPUT_DIR / "test_predictions_revised.pkl"
    )
    test_metrics_frame
else:
    print(
        "Test保持锁定。确认研究设计完全冻结后，"
        "手动将RUN_FINAL_TEST设为True并只运行一次。"
    )


## 运行后检查清单

- Validation时间必须晚于Train，Test时间必须晚于Validation。
- 任何参数、特征或阈值调整都不得使用Test。
- 记录 `lgb_model.best_iteration`、数据版本和特征清单。
- Validation结果只用于研究决策，不表述为最终样本外结果。
- 本Notebook不保证优化后预测或收益提高。
